# colab-bridge：把本 Colab 变成 AI 的远程执行后端本 notebook 做三件事：① 启动桥接服务（HTTP，端口 8787）② 启动外网隧道 ③ 打印连接配置。- 执行“从头运行”后依次跑完所有 cell。- **token 每次会话随机**，以第一次运行时打印的为准。- 免费版会断连（12h 上限 / 90min 空闲），断后重连需重跑本 notebook。- 模型文件会随会话丢失；要持久化请自行挂载 Drive 并把权重放 Drive。

In [ ]:
# [1/4] 写入桥接服务源码（自包含，零依赖）%%writefile colab_bridge.py#!/usr/bin/env python3# -*- coding: utf-8 -*-"""colab_bridge.py —— 在 Google Colab notebook 中运行的桥接服务。把 Colab 变成 pi（或任何本地程序）的远程 Python/GPU 执行后端：  POST /exec      {"mode":"python|bash","code":"...","timeout":N,"cwd":"..."} -> {exit,stdout,stderr,elapsed}  POST /upload    {"path":"/content/x.py","data":"<base64>"}                  -> {ok}  GET  /download?path=/content/x.txt                                         -> 文件字节  GET  /          -> 健康检查 + GPU 摘要（含 token）认证：请求头 X-Token（或 Authorization: Bearer <token>）。token 每次启动随机生成并打印。仅 stdlib，零依赖：Colab 里直接跑。"""import base64import jsonimport osimport secretsimport subprocessimport sysimport timeimport urllib.parsefrom http.server import BaseHTTPRequestHandler, HTTPServerTOKEN = os.environ.get("BRIDGE_TOKEN") or secrets.token_urlsafe(16)PORT = int(os.environ.get("BRIDGE_PORT", "8787"))MAX_OUT = 30000  # 单端 stdout/stderr 截断字符数def gpu_summary():    try:        r = subprocess.run(            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",             "--format=csv,noheader,nounits"],            capture_output=True, text=True, timeout=10)        return r.stdout.strip() or "-"    except Exception:        return "-"class Handler(BaseHTTPRequestHandler):    def log_message(self, *args):        pass    # ---------- 工具 ----------    def authed(self):        h = self.headers        return (h.get("X-Token") == TOKEN or                h.get("Authorization", "").replace("Bearer ", "") == TOKEN)    def reply(self, obj, code=200):        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")        self.send_response(code)        self.send_header("Content-Type", "application/json")        self.send_header("Content-Length", str(len(body)))        self.end_headers()        self.wfile.write(body)    def reply_bytes(self, data, ctype="application/octet-stream"):        self.send_response(200)        self.send_header("Content-Type", ctype)        self.send_header("Content-Length", str(len(data)))        self.end_headers()        self.wfile.write(data)    def read_json(self):        try:            ln = int(self.headers.get("Content-Length", 0))            return json.loads(self.rfile.read(ln))        except Exception:            return None    # ---------- GET ----------    def do_GET(self):        if not self.authed():            return self.reply({"error": "unauthorized"}, 401)        p = urllib.parse.urlparse(self.path)        if p.path == "/":            return self.reply({                "status": "ok",                "token": TOKEN,                "gpu": gpu_summary(),                "python": sys.version.split()[0],                "cwd": os.getcwd(),                "time": time.strftime("%Y-%m-%d %H:%M:%S %Z"),            })        if p.path == "/download":            q = urllib.parse.parse_qs(p.query)            path = q.get("path", [""])[0]            if not path or not os.path.exists(path):                return self.reply({"error": "not found"}, 404)            with open(path, "rb") as f:                return self.reply_bytes(f.read())        return self.reply({"error": "unknown path"}, 404)    # ---------- POST ----------    def do_POST(self):        if not self.authed():            return self.reply({"error": "unauthorized"}, 401)        p = urllib.parse.urlparse(self.path)        req = self.read_json()        if req is None:            return self.reply({"error": "bad json"}, 400)        if p.path == "/exec":            code = req.get("code", "")            mode = req.get("mode", "python")            timeout = float(req.get("timeout", 120))            cwd = req.get("cwd") or os.getcwd()            if mode == "bash":                cmd = ["bash", "-lc", code]            else:                cmd = [sys.executable, "-c", code]            t0 = time.time()            try:                r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout,                                   cwd=cwd, env={**os.environ, "PYTHONUNBUFFERED": "1"},                                   errors="replace")                return self.reply({                    "exit": r.returncode,                    "stdout": r.stdout[-MAX_OUT:],                    "stderr": r.stderr[-MAX_OUT:],                    "elapsed": round(time.time() - t0, 2),                })            except subprocess.TimeoutExpired:                return self.reply({"error": "timeout",                                   "elapsed": round(time.time() - t0, 2)}, 408)        if p.path == "/upload":            path = req.get("path", "")            data = req.get("data", "")            if not path:                return self.reply({"error": "path required"}, 400)            os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)            raw = base64.b64decode(data)            with open(path, "wb") as f:                f.write(raw)            return self.reply({"ok": True, "path": path, "bytes": len(raw)})        return self.reply({"error": "unknown path"}, 404)if __name__ == "__main__":    print(f"[bridge] listening on 0.0.0.0:{PORT}  token={TOKEN}", flush=True)    print(f"[bridge] GPU: {gpu_summary()}", flush=True)    HTTPServer(("0.0.0.0", PORT), Handler).serve_forever()

In [ ]:
# [2/4] 启动桥接服务（后台，跟随内核存活）import os, subprocess, sys, secrets, time, urllib.requestTOKEN = secrets.token_urlsafe(16)log = open("bridge.log", "w")proc = subprocess.Popen([sys.executable, "colab_bridge.py"],                        env={**os.environ, "BRIDGE_TOKEN": TOKEN, "BRIDGE_PORT": "8787"},                        stdout=log, stderr=subprocess.STDOUT)time.sleep(2)ok = Falsetry:    r = urllib.request.urlopen(f"http://127.0.0.1:8787/", headers={"X-Token": TOKEN}, timeout=5)    info = json.loads(r.read())    ok = Trueexcept Exception as e:    print("启动失败:", e); print(open("bridge.log").read()[-2000:])print("TOKEN =", TOKEN)if ok:    print(f"[bridge] ok  GPU: {info.get('gpu')}  Python: {info.get('python')}")

In [ ]:
# [3/4] 外网隧道：cloudflared 快速隧道（免账号，国内网络如不稳可换下方 tailscale）# 第一次运行会下载二进制（~50MB），之后复用。import os, subprocess, time, urllib.request, pathlibCF = "cloudflared"if not pathlib.Path(CF).exists():    subprocess.run(["curl", "-fsSL", "-o", CF,        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],        check=True)    os.chmod(CF, 0o755)log = open("tunnel.log", "w")t = subprocess.Popen([os.path.abspath(CF), "tunnel", "--url", "http://localhost:8787"],                     stdout=log, stderr=subprocess.STDOUT)url = Nonefor _ in range(30):    time.sleep(2)    txt = open("tunnel.log", errors="replace").read()    import re    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)    if m:        url = m.group(0)        breakif url:    print("CLOUD_URL =", url)else:    print("隧道未就绪，看日志："); print(open("tunnel.log").read()[-1500:])

居备用：若 cloudflared 不通，可用 tailscale（需 GitHub/Google 账号登录一次，之后 IP 固定）：```python# !curl -fsSL https://tailscale.com/install.sh | sh# !tailscale up   # 按提示打开链接授权# !tailscale ip -4   # 设备在 tailnet 内的 IP；从本机 tailscale 网络访问 <ip>:8787```

In [ ]:
# [4/4] 自检 + 打印连接配置import json, urllib.requesttry:    r = urllib.request.urlopen("http://127.0.0.1:8787/", headers={"X-Token": TOKEN}, timeout=5)    print(json.loads(r.read()))except Exception as e:    print("bridge 未响应:", e)print()print("================ 复制保存以下配置 ================")print(json.dumps({"url": globals().get("url","<CLOUD_URL 或 tailscale IP:8787>"), "token": TOKEN}, ensure_ascii=False))print("==================================================")